Run the cells in this section to install the packages needed by the notebooks in this workshop.

IGNORE ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.

In [2]:
%pip install --no-build-isolation --force-reinstall \
    "boto3>=1.28.57" \
    "awscli>=1.29.57" \
    "botocore>=1.31.57"


  Using cached boto3-1.42.20-py3-none-any.whl.metadata (6.8 kB)
  Using cached awscli-1.44.10-py3-none-any.whl.metadata (11 kB)
  Using cached botocore-1.42.20-py3-none-any.whl.metadata (5.9 kB)
  Using cached jmespath-1.0.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached s3transfer-0.16.0-py3-none-any.whl.metadata (1.7 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.2-py3-none-any.whl.metadata (6.6 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached docutils-0.19-py3-none-any.whl.metadata (2.7 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached rsa-4.7.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached pyasn1-0.6.1-py3-none-any.whl.metadata (8.4 kB)
Using cached boto3-1.42.20-py3-none-any.whl (140 kB)
Using cached botocore-1.42

This notebook demonstrates invoking Bedrock models directly using the AWS SDK, but for later part of this notebook, you'll also need to install other packages

In [3]:
%pip install  \
    "langchain>=0.0.350" \
    "transformers>=4.24,<5" \
    sqlalchemy -U \
    "faiss-cpu>=1.7,<2" \
    "pypdf>=3.8,<4" \
    pinecone-client==2.2.4 \
    tiktoken==0.5.2 \
    "ipywidgets>=7,<8" \
    matplotlib==3.8.2 \
    anthropic==0.9.0 \
    datasets==2.15.0 \
    numexpr==2.8.8

Note: you may need to restart the kernel to use updated packages.


### Restart Kernel 

In [4]:
# restart kernel
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

The boto3 provides different clients for Amazon Bedrock to perform different actions. The actions for InvokeModel and InvokeModelWithResponseStream are supported by Amazon Bedrock Runtime

In [5]:
import json
import os
import sys

import boto3

module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import bedrock



boto3_bedrock = bedrock.get_bedrock_client(
    runtime = True
)

Create new client
  Using region: us-east-1
boto3 Bedrock client successfully created!
bedrock-runtime(https://bedrock-runtime.us-east-1.amazonaws.com)


In [6]:
pip install -U langchain-community

Note: you may need to restart the kernel to use updated packages.


In [7]:
# We will be using the Titan Embeddings Model to generate our Embeddings.
from langchain_community.embeddings import BedrockEmbeddings
from langchain_community.llms import Bedrock

# - create the Anthropic Model
llm = Bedrock(model_id="anthropic.claude-v2", client=boto3_bedrock, model_kwargs={'max_tokens_to_sample':300})
bedrock_embeddings = BedrockEmbeddings(model_id="amazon.titan-embed-text-v1", client=boto3_bedrock)

/tmp/ipykernel_5846/1718514090.py:6: LangChainDeprecationWarning: The class `Bedrock` was deprecated in LangChain 0.0.34 and will be removed in 1.0. An updated version of the class exists in the `langchain-aws package and should be used instead. To use it run `pip install -U `langchain-aws` and import as `from `langchain_aws import BedrockLLM``.
  llm = Bedrock(model_id="anthropic.claude-v2", client=boto3_bedrock, model_kwargs={'max_tokens_to_sample':300})
/tmp/ipykernel_5846/1718514090.py:7: LangChainDeprecationWarning: The class `BedrockEmbeddings` was deprecated in LangChain 0.2.11 and will be removed in 1.0. An updated version of the class exists in the `langchain-aws package and should be used instead. To use it run `pip install -U `langchain-aws` and import as `from `langchain_aws import BedrockEmbeddings``.
  bedrock_embeddings = BedrockEmbeddings(model_id="amazon.titan-embed-text-v1", client=boto3_bedrock)


We begin with instantiating the LLM and the Embeddings model. Here we are using Anthropic Claude for text generation and Amazon Titan for text embedding.

Note: It is possible to choose other models available with Bedrock. You can replace the model_id as follows to change the model.

Let's first download some of the files to build our document store. In this example I am downloading the official paper for RAG.

After downloading we can load the documents with the help of DirectoryLoader from PyPDF available under LangChain and splitting them into smaller chunks.

Note: The retrieved document/text should be large enough to contain enough information to answer a question; but small enough to fit into the LLM prompt. 
Also the embeddings model has a limit of the length of input tokens limited to 8192 tokens, which roughly translates to ~32,000 characters. 
For the sake of this use-case we are creating chunks of roughly 2000 characters with an overlap of 200 characters using RecursiveCharacterTextSplitter.


In [8]:
pip install langchain-text-splitters 

Note: you may need to restart the kernel to use updated packages.


In [9]:
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader

documents = WebBaseLoader("https://www.frbsf.org/our-people/leadership/",header_template={"User-Agent":"Mozilla/5.0 (compatible; MyBot/1.0)"}).load()
# - in our testing Character split works better with this PDF data set
text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size = 2000,
    chunk_overlap  = 200,
)
docs = text_splitter.split_documents(documents)

USER_AGENT environment variable not set, consider setting it to identify your requests.


Lets review how many chunks and characters we are dealing with

In [10]:

abs
avg_doc_length = lambda documents: sum([len(doc.page_content) for doc in documents])//len(documents)
avg_char_count_pre = avg_doc_length(documents)
avg_char_count_post = avg_doc_length(docs)
print(f'Average length among {len(documents)} documents loaded is {avg_char_count_pre} characters.')
print(f'After the split we have {len(docs)} documents more than the original {len(documents)}.')
print(f'Average length among {len(docs)} documents (after split) is {avg_char_count_post} characters.')

Average length among 1 documents loaded is 4272 characters.
After the split we have 3 documents more than the original 1.
Average length among 3 documents (after split) is 1516 characters.


Now we can see how a sample embedding would look like for the first chunk

In [11]:
try:
    
    sample_embedding = np.array(bedrock_embeddings.embed_query(docs[1].page_content))
    print("Sample chunk: ",docs[0].page_content)
    print("Sample embedding of a document chunk: ", sample_embedding)
    print("Size of the embedding: ", sample_embedding.shape)

except ValueError as error:
    if  "AccessDeniedException" in str(error):
        print(f"\x1b[41m{error}\
        \nTo troubeshoot this issue please refer to the following resources.\
         \nhttps://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_access-denied.html\
         \nhttps://docs.aws.amazon.com/bedrock/latest/userguide/security-iam.html\x1b[0m\n")      
        class StopExecution(ValueError):
            def _render_traceback_(self):
                pass
        raise StopExecution        
    else:
        raise error

Sample chunk:  Leader­­­­ship - San Francisco Fed





































 
















Federal Reserve Bank of San Francisco

Federal Reserve Bank of San Francisco

















LinkedIn








Facebook







X








YouTube






Instagram







Threads






About Us
Our People
Join Us






What We Study


 Back
What We Study


Monetary Policy


Labor Markets


Inflation


US Economy


Global Economy


Banking


Financial Markets


Technology


View All Topics




Our District


 Back
Our District


Alaska


Arizona


California


Hawai’i


Idaho


Nevada


Oregon


Utah


Washington


Territories and Commonwealth


Native Communities




Research & Insights


 Back
Research & Insights


Economic Research


Community Engagement and Analysis


EmergingTech Economic Research Network (EERN)


Data & Indicators


Blog Posts


SF Fed Publications




News & Media


 Back
News & Media


Events


Speeches


News


Zip Code Economies


Subscrip­tions





Search:



Following the similar pattern embeddings could be generated for the entire corpus and stored in a vector store.

This can be easily done using FAISS implementation inside LangChain which takes input the embeddings model and the documents to create the entire vector store. 
Using the Index Wrapper we can abstract away most of the heavy lifting such as creating the prompt, getting embeddings of the query, sampling the relevant documents and calling the LLM. 
VectorStoreIndexWrapper helps us with that.

⚠️⚠️⚠️ NOTE: it might take few minutes to run the following cell ⚠️⚠️⚠️


In [12]:
pip install -U langchain-core

Note: you may need to restart the kernel to use updated packages.


In [13]:

from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.faiss import DistanceStrategy
#from langchain_community.indexes import VectorstoreIndexCreator
#from langchain_community.vectorstores import VectorStoreIndexWrapper

vectorstore_faiss = FAISS.from_documents(
    docs,
    bedrock_embeddings,
    distance_strategy=DistanceStrategy.COSINE
)

#wrapper_store_faiss = VectorStoreIndexWrapper(vectorstore=vectorstore_faiss)

Now that we have our vector store in place, we can start asking questions.

In [14]:
query = """Who is Mary C.Daly"""

Lets review the mebedding for the query.

In [15]:
query_embedding = vectorstore_faiss.embedding_function.embed_query(query)
np.array(query_embedding)

array([ 0.19433594, -0.5703125 ,  0.24804688, ...,  0.24414062,
       -0.75      ,  0.49414062])

We can use this embedding of the query to then fetch relevant documents. Now our query is represented as embeddings we can do a similarity search of our query against our data store providing us with the most relevant information.

In [16]:
relevant_documents = vectorstore_faiss.similarity_search_by_vector(query_embedding)
print(f'{len(relevant_documents)} documents are fetched which are relevant to the query.')
print('----')
for i, rel_doc in enumerate(relevant_documents):
    print(f'## Document {i+1}: {rel_doc.page_content}.......')
    print('---')

3 documents are fetched which are relevant to the query.
----
## Document 1: Leader­­­­ship - San Francisco Fed





































 
















Federal Reserve Bank of San Francisco

Federal Reserve Bank of San Francisco

















LinkedIn








Facebook







X








YouTube






Instagram







Threads






About Us
Our People
Join Us






What We Study


 Back
What We Study


Monetary Policy


Labor Markets


Inflation


US Economy


Global Economy


Banking


Financial Markets


Technology


View All Topics




Our District


 Back
Our District


Alaska


Arizona


California


Hawai’i


Idaho


Nevada


Oregon


Utah


Washington


Territories and Commonwealth


Native Communities




Research & Insights


 Back
Research & Insights


Economic Research


Community Engagement and Analysis


EmergingTech Economic Research Network (EERN)


Data & Indicators


Blog Posts


SF Fed Publications




News & Media


 Back
News & Media


Events


Speech

You have the possibility to use the wrapper provided by LangChain which wraps around the Vector Store and takes input the LLM. This wrapper performs the following steps behind the scences:

    Take the question as input
    Create question embedding
    Fetch relevant documents
    Stuff the documents and the question into a prompt
    Invoke the model with the prompt and generate the answer in a human readable manner.


Lets ask a question which cannot be answewred on the the bases of provided content

You can also query the vector database and find the similarity score. the lower the score is, the better the result is. Read more about this https://python.langchain.com/docs/integrations/vectorstores/faiss

In [ ]:
db = FAISS.from_documents(
    docs,
    bedrock_embeddings,
    distance_strategy=DistanceStrategy.COSINE # This line sets the configuration
)

print("FAISS vector database created with COSINE distance configuration.")

# 4. Example search with scores (scores will be 0-1, higher is better)
query = "Who is Mary C.Daly"
results_with_scores = db.similarity_search_with_score(query, k=2)

print(f"\nSearch results for query: '{query}'\n")

for document, score in results_with_scores:
    print("-" * 40)
    # When using COSINE strategy, LangChain maps distance [0, 2] to a relevance score [0, 1]
    print(f"Relevance Score (0-1, higher is better): {score:.4f}")
    print(f"Document snippet: {document.page_content}")

In [19]:
import boto3
from langchain_aws import BedrockEmbeddings, ChatBedrockConverse

bedrock_client = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1" # Use your region
)

# Initialize the Bedrock LLM (Example using Claude 3 Haiku)

# Initialize the Bedrock LLM using ChatBedrockConverse (which uses Messages API)
llm = ChatBedrockConverse(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0", # The model ID that caused the error previously
    client=bedrock_client,
    max_tokens=512,        # <-- Pass max_tokens directly
    temperature=0.1 
    # Note: Use 'max_tokens' instead of 'max_tokens_to_sample' for new models/APIs
    #model_kwargs={"max_tokens": 512, "temperature": 0.1}
)

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = db.as_retriever(search_kwargs={"k": 3})


# --- 3. Define the RAG Prompt Template (Optimized for Chat Models) ---

# Chat models prefer system prompts and specific message roles.
# Use MessagesPlaceholder for more complex chat history handling if needed,
# but for basic RAG, we stick to system/human roles.
rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful AI assistant.\n\nContext: {context}",
        ),
        ("human", "{question}"),
    ]
)


# --- 4. Build and Run the RAG Chain ---
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

question = "Who is Mary C.Daly?"
print(f"Asking question: {question}\n")

# Invoke the chain
response = rag_chain.invoke(question)

print("Response from Bedrock LLM:")
print(response)

Asking question: Who is Mary C.Daly?

Response from Bedrock LLM:
Based on the provided context, Mary C. Daly is the President and Chief Executive Officer of the Federal Reserve Bank of San Francisco. 

The Federal Reserve Bank of San Francisco is one of the 12 Reserve Banks in the Federal Reserve System and serves as the headquarters of the Twelfth Federal Reserve District, which includes nine western states (Alaska, Arizona, California, Hawaii, Idaho, Nevada, Oregon, Utah, and Washington) plus American Samoa, Guam, and the Commonwealth of the Northern Mariana Islands.

As President and CEO, she leads one of the key regional Federal Reserve banks in the United States financial system.


In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = db.as_retriever(search_kwargs={"k": 3})


# --- 3. Define the RAG Prompt Template (Optimized for Chat Models) ---

# Chat models prefer system prompts and specific message roles.
# Use MessagesPlaceholder for more complex chat history handling if needed,
# but for basic RAG, we stick to system/human roles.
rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful AI assistant.\n\nContext: {context}",
        ),
        ("human", "{question}"),
    ]
)


# --- 4. Build and Run the RAG Chain ---
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

question = "Who is elon musk?"
print(f"Asking question: {question}\n")

# Invoke the chain
response = rag_chain.invoke(question)

print("Response from Bedrock LLM:")
print(response)

Asking question: Who is elon musk?

Response from Bedrock LLM:
I don't see any information about Elon Musk in the provided context, which appears to be about the Federal Reserve Bank of San Francisco's leadership team and organizational structure.

Based on general knowledge, Elon Musk is a prominent entrepreneur and business magnate known for:

- Being the CEO of Tesla, the electric vehicle and clean energy company
- Being the owner and CTO of X (formerly Twitter)
- Being the founder and CEO of SpaceX, a private space exploration company
- Being involved in other ventures like Neuralink, The Boring Company, and previously PayPal

However, since this information isn't contained in the provided context about the San Francisco Fed, I should note that my response is based on general knowledge rather than the specific documents you've shared. Is there something specific about the Federal Reserve Bank of San Francisco's leadership that you'd like to know about instead?
